# For HGERE the batch size for the GNN is dependend on the number of entities in each sentence.
  * Create a simplified version of the dataset
  * sett the maximum number of ners per sentence to a fixed number (e.g., 18)

In [37]:
from glob import glob
import json

In [38]:
dataset_files = glob("../../../gsap-rel/preprocessing/gsap-rel-sentence-simple/*.jsonl")

In [40]:
ner_length = []
ner_length_truncated = []
skipped_rels = 0
skipped_ents = 0
max_ents_per_sentence = 18
for fn in dataset_files:
    target_path = f"{fn[:-6]}-truncated.jsonl"
    print(target_path)
    with open(target_path, 'w') as f:
        for idx, line in enumerate(open(fn).readlines()):
            doc = json.loads(line)
            ner = doc["ner"]
            ner_truncated = []
            for sent_ner in ner:
                ner_truncated.append(sent_ner[:max_ents_per_sentence])
                skipped_ents += len(sent_ner[max_ents_per_sentence:])
                ner_length.append(len(sent_ner))
            doc["ner"] = ner_truncated
            ner_keys = {(begin, end) for sent_ner in ner_truncated for begin, end, _ in sent_ner}
            rels = doc["relations"]
            rels_truncated = []
            for sent_rel in rels:
                sent_rels_truncated = []
                for begin, end, begin_o, end_o, label in sent_rel:
                    if (begin, end) in ner_keys and (begin_o, end_o) in ner_keys:
                        sent_rels_truncated.append((begin, end, begin_o, end_o, label))
                    else:
                        skipped_rels += 1
                rels_truncated.append(sent_rels_truncated)
            doc["relations"] = rels_truncated
            doc_json = json.dumps(doc)
            if idx > 0:
                doc_json = "\n" + doc_json
            f.write(doc_json)

../../../gsap-rel/preprocessing/gsap-rel-sentence-simple/train-truncated.jsonl
../../../gsap-rel/preprocessing/gsap-rel-sentence-simple/dev-truncated.jsonl
../../../gsap-rel/preprocessing/gsap-rel-sentence-simple/test-truncated.jsonl
../../../gsap-rel/preprocessing/gsap-rel-sentence-simple/train-truncated-truncated.jsonl
../../../gsap-rel/preprocessing/gsap-rel-sentence-simple/dev-truncated-truncated.jsonl
../../../gsap-rel/preprocessing/gsap-rel-sentence-simple/test-truncated-truncated.jsonl
../../../gsap-rel/preprocessing/gsap-rel-sentence-simple/train_debug-truncated.jsonl
../../../gsap-rel/preprocessing/gsap-rel-sentence-simple/dev_debug-truncated.jsonl
../../../gsap-rel/preprocessing/gsap-rel-sentence-simple/test_debug-truncated.jsonl


In [41]:
skipped_ents, skipped_rel

(232, 0)

In [33]:
ner_keys_all = {(begin, end) for sent_ner in ner for begin, end, _ in sent_ner}
len(ner_keys_all)

737

In [24]:
from itertools import chain
len(list(chain(*ner)))

756

In [23]:
len(ner_keys)

737

In [26]:
import pandas as pd

In [27]:
pd.Series(ner_length).value_counts().sort_index()

0     7412
1     5282
2     4556
3     3329
4     2404
5     1557
6      903
7      553
8      348
9      225
10     132
11      83
12      54
13      58
14      32
15      25
16      16
17      16
18      11
19       5
20       7
21       4
22       1
23       2
24       1
25       2
26       1
30       1
32       1
35       1
36       2
38       1
44       1
52       1
dtype: int64